# Fold-aware public U-Net crossfit evaluation

Run the shared EquiNexus evaluator on supplied competition training images and the public 10-model U-Net checkpoint dataset. For each scored observation, it uses only the two checkpoints whose upstream fold excluded that observation. The official scorer and frozen seed-42 observation split are unchanged.

The upstream folds partition all 707 training observations. Other observations from the 142-observation frozen-holdout subset can therefore appear in the training data for other fold models. Report this as per-observation crossfit on the frozen-holdout sample, not as an independent frozen holdout or promotion gate. The run reads no public MAGFiLO test labels and does not predict on test images.

The model dataset is licensed CC BY-NC-SA 4.0. Preserve its attribution and ShareAlike terms.

In [ ]:
from pathlib import Path
import glob, importlib.metadata, importlib.util, json, os, subprocess, sys

REPOSITORY_URL = "https://github.com/BrownAssassin/EquiNexus.git"
REPOSITORY_DIR = Path("/kaggle/working/EquiNexus")
OUTPUT_DIR = Path("/kaggle/working/dariushafshar-unet-fold-aware-holdout")
SPLIT_DIR = Path("/kaggle/working/equinexus-splits")

def exactly_one(pattern, description):
    matches = sorted(glob.glob(pattern, recursive=True))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {description}, found {len(matches)}: {matches[:5]}")
    return Path(matches[0])

annotation_path = exactly_one("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", "competition training annotation file")
data_root = annotation_path.parents[1]
receipt_path = exactly_one("/kaggle/input/**/solar_unet_receipt.json", "public U-Net receipt")
weights_root = receipt_path.parent
wheel_path = exactly_one("/kaggle/input/**/segmentation_models_pytorch-0.5.0-py3-none-any.whl", "SMP 0.5.0 wheel")

if importlib.util.find_spec("segmentation_models_pytorch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel_path)], check=True)
try:
    timm_version = importlib.metadata.version("timm")
except importlib.metadata.PackageNotFoundError:
    timm_version = None
if timm_version != "1.0.29":
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "timm==1.0.29"], check=True)

if REPOSITORY_DIR.exists():
    raise RuntimeError(f"Refusing to overwrite {REPOSITORY_DIR}")
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
source_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True).stdout.strip()
print(json.dumps({"source_commit": source_commit, "data_root": str(data_root), "weights_root": str(weights_root), "output_dir": str(OUTPUT_DIR), "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()}, indent=2))

In [ ]:
command = [
    sys.executable, str(REPOSITORY_DIR / "scripts" / "evaluate_public_unet_fold_aware.py"),
    "--data-root", str(data_root),
    "--weights-root", str(weights_root),
    "--split-dir", str(SPLIT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda:0",
    "--tta-views", "8",
    "--views-per-pass", "1",
]
subprocess.run(command, cwd=REPOSITORY_DIR, check=True)

In [ ]:
metadata_path = OUTPUT_DIR / "run-metadata.json"
run = json.loads(metadata_path.read_text(encoding="utf-8"))
print(json.dumps({
    "source_commit": run["source_commit"],
    "split_sha256": run["competition_split_sha256"],
    "validation_observations": run["validation_observations"],
    "annotator_views": run["validation_annotator_views"],
    "official_pq": run["metrics"]["global"]["pq"],
    "tp_fp_fn": [run["metrics"]["global"][k] for k in ("tp", "fp", "fn")],
    "runtime_seconds": run["inference_seconds"],
    "peak_gpu_memory_allocated_bytes": run["peak_gpu_memory_allocated_bytes"],
}, indent=2))